In [ ]:
!pip install earthengine-api --quiet
!pip install folium --quiet
!pip install geemap --quiet
!pip install tensorflow --quiet

In [ ]:
dataset = "easy"


# Earth Engine & Mapping
import ee
import geemap
import folium
from folium.plugins import MarkerCluster

# Core Python
import json
from datetime import datetime, timedelta

# Data Handling
import pandas as pd
import numpy as np

ee.Authenticate()
ee.Initialize(project='ee-enisbirgili')

In [ ]:
# Step 1: Define AOI
if dataset == "easy":
  aoi = ee.Geometry.Rectangle([
      -90.33133359168521,  # min longitude (west)
      35.56839127741864,   # min latitude (south)
      -90.28652997230044,  # max longitude (east)
      35.59826718872311    # max latitude (north)
  ])
if dataset == "hard":
  aoi = ee.Geometry.Rectangle([
      -77.53475507131758,  # min longitude (west)
      35.924430554035496,   # min latitude (south)
      -77.72152264944258,  # max longitude (east)
      36.047219811319366  # max latitude (north)
  ])

In [ ]:
# Step 2: Load USDA CDL crop map for 2024
cdl = ee.ImageCollection("USDA/NASS/CDL") \
    .filterDate("2024-01-01", "2024-12-31") \
    .first()

"""
# Step 3: Filter CDL to crop codes 1 (Corn) and 5 (Soybeans)
valid_crops = ee.List([1, 5])
cdl_masked = cdl.updateMask(cdl.remap(valid_crops, [1, 1]))  # Only keep 1 and 5
"""
cdl_masked = cdl

# Step 4: Sample labeled CDL points
crop_classes = [1, 2, 5, 10, 11, 26, 46, 142]
points_per_class = 150  # Adjust depending on your use case and balance needs

# Stratified sampling using 'cropland' band
labelled_points = cdl.stratifiedSample(
    numPoints=points_per_class,
    classBand='cropland',
    classValues=crop_classes,
    classPoints=[points_per_class] * len(crop_classes),
    region=aoi,
    scale=30,
    seed=42,
    geometries=True
)

print("Sample collection started... (not using getInfo to avoid timeouts)")

# Step 5: Load Sentinel-2 data (non-deprecated)
s2 = ee.ImageCollection("COPERNICUS/S2_HARMONIZED") \
    .filterBounds(aoi) \
    .filterDate("2024-03-01", "2024-10-31") \
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)) \
    .select(["B2", "B3", "B4", "B8"])  # Blue, Green, Red, NIR

# Step 6: Add NDVI to each image
def add_ndvi(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return img.addBands(ndvi)

s2 = s2.map(add_ndvi)

# Step 7: Sample each Sentinel-2 image at labeled points
def sample_at_points(img):
    sampled = img.sampleRegions(
        collection=labelled_points,
        scale=30,
        geometries=True
    ).map(lambda f: f.set("date", img.date().format("YYYY-MM-dd")))
    return sampled

sampled_collection = s2.map(sample_at_points).flatten()

# Step 8: Export to Google Drive (don't use .getInfo() on big data!)
task = ee.batch.Export.table.toDrive(
    collection=sampled_collection,
    description='crops',
    fileFormat='CSV'
)

task.start()
print("Export started. Check Tasks tab in Earth Engine or Google Cloud Console.")

Sample collection started... (not using getInfo to avoid timeouts)
Export started. Check Tasks tab in Earth Engine or Google Cloud Console.
